# Table of Contents
### 1. Importing libraries and data
### 2. Data Cleaning and Wrangling
### 3. Reshaping for Modeling
### 4. Splitting the data
### 5. Optimizing Bayesian Hyperparameters
### 6. Running the CNN with new Parameters
### 7. Creating the Confusion Matrix

# 1. Importing libraries and data

In [7]:
import pandas as pd
import numpy as np
import seaborn as sns
import os
import operator
import time
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from numpy import unique
from numpy import reshape
from keras.models import Sequential
from sklearn.model_selection import cross_val_score
from keras.layers import Conv1D, Conv2D, Dense, Dropout, BatchNormalization, Flatten, MaxPooling1D
from tensorflow.keras.utils import to_categorical
from keras.optimizers import Adam, SGD, RMSprop, Adadelta, Adagrad, Adamax, Nadam, Ftrl
from keras.callbacks import EarlyStopping, ModelCheckpoint
from scikeras.wrappers import KerasClassifier
from math import floor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import make_scorer, accuracy_score
from bayes_opt import BayesianOptimization
from sklearn.model_selection import StratifiedKFold
from keras.layers import LeakyReLU
LeakyReLU = LeakyReLU(alpha=0.1)

C:\Users\andyc\anaconda3\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


In [9]:
# Identifying a path to where data is stored

path = 'C:/Users/andyc/Machine Learning - ClimateWins'

In [11]:
# Importing the cleaned weather data set

cleancw = pd.read_csv(os.path.join(path, 'Data Sets', 'cleaned_weather.csv'), index_col = False)

In [13]:
# Importing the pleasant_weather data set

pleasant = pd.read_csv(os.path.join(path, 'Data Sets', 'Dataset-Answers-Weather_Prediction_Pleasant_weather.csv'), index_col = False)

In [15]:
cleancw.head()

,DATE,MONTH,BASEL_cloud_cover,BASEL_humidity,BASEL_pressure,BASEL_global_radiation,BASEL_precipitation,BASEL_sunshine,BASEL_temp_mean,BASEL_temp_min,...,STOCKHOLM_temp_max,VALENTIA_cloud_cover,VALENTIA_humidity,VALENTIA_pressure,VALENTIA_global_radiation,VALENTIA_precipitation,VALENTIA_sunshine,VALENTIA_temp_mean,VALENTIA_temp_min,VALENTIA_temp_max
0,19600101,1,7,0.85,1.018,0.32,0.09,0.7,6.5,0.8,...,4.9,5,0.88,1.0003,0.45,0.34,4.7,8.5,6.0,10.9
1,19600102,1,6,0.84,1.018,0.36,1.05,1.1,6.1,3.3,...,5.0,7,0.91,1.0007,0.25,0.84,0.7,8.9,5.6,12.1
2,19600103,1,8,0.90,1.018,0.18,0.30,0.0,8.5,5.1,...,4.1,7,0.91,1.0096,0.17,0.08,0.1,10.5,8.1,12.9
3,19600104,1,3,0.92,1.018,0.58,0.00,4.1,6.3,3.8,...,2.3,7,0.86,1.0184,0.13,0.98,0.0,7.4,7.3,10.6
4,19600105,1,6,0.95,1.018,0.65,0.14,5.4,3.0,-0.7,...,4.3,3,0.80,1.0328,0.46,0.00,5.7,5.7,3.0,8.4


In [17]:
pleasant.head()

,DATE,BASEL_pleasant_weather,BELGRADE_pleasant_weather,BUDAPEST_pleasant_weather,DEBILT_pleasant_weather,DUSSELDORF_pleasant_weather,HEATHROW_pleasant_weather,KASSEL_pleasant_weather,LJUBLJANA_pleasant_weather,MAASTRICHT_pleasant_weather,MADRID_pleasant_weather,MUNCHENB_pleasant_weather,OSLO_pleasant_weather,SONNBLICK_pleasant_weather,STOCKHOLM_pleasant_weather,VALENTIA_pleasant_weather
0,19600101,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,19600102,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,19600103,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,19600104,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,19600105,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [19]:
cleancw.shape

(22950, 137)

In [21]:
pleasant.shape

(22950, 16)

# 2. Data Cleaning and Wrangling

In [24]:
# Dropping the unnecessary columns

cleancw.drop(['DATE', 'MONTH'], axis=1, inplace=True)

In [26]:
cleancw.shape

(22950, 135)

In [28]:
pleasant.drop(columns = 'DATE', inplace = True)

In [30]:
pleasant.shape

(22950, 15)

# 3. Reshaping for Modeling

In [35]:
# Turning cleancw and pleasant dataframes from dataframes to arrays

X = np.array(cleancw)
y = np.array(pleasant)

In [37]:
X = X.reshape(-1,15,9)

In [39]:
X.shape

(22950, 15, 9)

In [41]:
# Using argmax to transform y

y =  np.argmax(y, axis = 1)
y

array([0, 0, 0, ..., 0, 0, 0], dtype=int64)

In [43]:
y.shape

(22950,)

In [45]:
# Checking y layout

from sklearn.utils.multiclass import type_of_target
type_of_target(y)

'multiclass'

Only 'multiclass' and 'binary' layouts are accepted by the Bayesian Optimization function.

# 4. Splitting the data

In [49]:
# Split data into train and test sets

X_train, X_test, y_train, y_test = train_test_split(X,y,random_state = 28)

In [51]:
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(17212, 15, 9) (17212,)
(5738, 15, 9) (5738,)


# 5. Optimizing Bayesian Hyperparameters

In [56]:
timesteps = len(X_train[0])
input_dim = len(X_train[0][0])
n_classes = 15 # Number of weather stations

# Make scorer accuracy
score_acc = make_scorer(accuracy_score)

In [58]:
# Create function

def bay_area(neurons, activation, kernel, optimizer, learning_rate, batch_size, epochs,
              layers1, layers2, normalization, dropout, dropout_rate): 
    optimizerL = ['SGD', 'Adam', 'RMSprop', 'Adadelta', 'Adagrad', 'Adamax', 'Nadam', 'Ftrl','SGD']
    #optimizerD= {'Adam':Adam(lr=learning_rate), 'SGD':SGD(lr=learning_rate),
                 #'RMSprop':RMSprop(lr=learning_rate), 'Adadelta':Adadelta(lr=learning_rate),
                 #'Adagrad':Adagrad(lr=learning_rate), 'Adamax':Adamax(lr=learning_rate),
                 #'Nadam':Nadam(lr=learning_rate), 'Ftrl':Ftrl(lr=learning_rate)}
    activationL = ['relu', 'sigmoid', 'softplus', 'softsign', 'tanh', 'selu',
                   'elu', 'exponential', LeakyReLU,'relu']
    
    neurons = round(neurons)
    kernel = round(kernel)
    activation = activationL[round(activation)]  #optimizerD[optimizerL[round(optimizer)]]
    optimizer = optimizerL[round(optimizer)]
    batch_size = round(batch_size)
    
    epochs = round(epochs)
    layers1 = round(layers1)
    layers2 = round(layers2)
    
    def cnn_model():
        model = Sequential()
        model.add(Conv1D(neurons, kernel_size=kernel,activation=activation, input_shape=(timesteps, input_dim)))
        #model.add(Conv1D(32, kernel_size=1,activation='relu', input_shape=(timesteps, input_dim)))
        
        if normalization > 0.5:
            model.add(BatchNormalization())
        for i in range(layers1):
            model.add(Dense(neurons, activation=activation)) #(neurons, activation=activation))
        if dropout > 0.5:
            model.add(Dropout(dropout_rate, seed=123))
        for i in range(layers2):
            model.add(Dense(neurons, activation=activation))
        model.add(MaxPooling1D())
        model.add(Flatten())
        model.add(Dense(n_classes, activation='softmax')) #sigmoid softmax
        #model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy']) #categorical_crossentropy
        model.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer, metrics=['accuracy']) #categorical_crossentropy
        return model
    es = EarlyStopping(monitor='accuracy', mode='max', verbose=2, patience=20)
    nn = KerasClassifier(build_fn=cnn_model, epochs=epochs, batch_size=batch_size, verbose=2)
    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
    score = cross_val_score(nn, X_train, y_train, scoring=score_acc, cv=kfold, fit_params={'callbacks':[es]}).mean()
    return score

In [60]:
start = time.time()
params ={
    'neurons': (10, 100),
    'kernel': (1, 3),
    'activation':(0, 9), 
    'optimizer':(0,7),
    'learning_rate':(0.01, 1),
    'batch_size': (200, 1000), 
    'epochs':(20, 50),
    'layers1':(1,3),
    'layers2':(1,3),
    'normalization':(0,1),
    'dropout':(0,1),
    'dropout_rate':(0,0.3)
}

# Run Bayesian Optimization
nn_opt = BayesianOptimization(bay_area, params, random_state=28)
nn_opt.maximize(init_points=15, n_iter=4) 
print('Search took %s minutes' % ((time.time() - start)/60))

|   iter    |  target   | activa... | batch_... |  dropout  | dropou... |  epochs   |  kernel   |  layers1  |  layers2  | learni... |  neurons  | normal... | optimizer |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------


C:\Users\andyc\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:73: FutureWarning: `fit_params` is deprecated and will be removed in version 1.6. Pass parameters via `params` instead.
  warnings.warn(
C:\Users\andyc\anaconda3\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
C:\Users\andyc\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/43
22/22 - 3s - 157ms/step - accuracy: 0.6442 - loss: nan
Epoch 2/43
22/22 - 1s - 55ms/step - accuracy: 0.6442 - loss: nan
Epoch 3/43
22/22 - 1s - 25ms/step - accuracy: 0.6442 - loss: nan
Epoch 4/43
22/22 - 1s - 27ms/step - accuracy: 0.6442 - loss: nan
Epoch 5/43
22/22 - 1s - 32ms/step - accuracy: 0.6442 - loss: nan
Epoch 6/43
22/22 - 1s - 28ms/step - accuracy: 0.6442 - loss: nan
Epoch 7/43
22/22 - 1s - 26ms/step - accuracy: 0.6442 - loss: nan
Epoch 8/43
22/22 - 1s - 30ms/step - accuracy: 0.6442 - loss: nan
Epoch 9/43
22/22 - 1s - 23ms/step - accuracy: 0.6442 - loss: nan
Epoch 10/43
22/22 - 1s - 25ms/step - accuracy: 0.6442 - loss: nan
Epoch 11/43
22/22 - 1s - 24ms/step - accuracy: 0.6442 - loss: nan
Epoch 12/43
22/22 - 1s - 25ms/step - accuracy: 0.6442 - loss: nan
Epoch 13/43
22/22 - 1s - 26ms/step - accuracy: 0.6442 - loss: nan
Epoch 14/43
22/22 - 1s - 27ms/step - accuracy: 0.6442 - loss: nan
Epoch 15/43
22/22 - 1s - 33ms/step - accuracy: 0.6442 - loss: nan
Epoch 16/43
22/22 

C:\Users\andyc\anaconda3\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
C:\Users\andyc\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/43
22/22 - 4s - 160ms/step - accuracy: 0.6143 - loss: nan
Epoch 2/43
22/22 - 1s - 25ms/step - accuracy: 0.6443 - loss: nan
Epoch 3/43
22/22 - 1s - 28ms/step - accuracy: 0.6443 - loss: nan
Epoch 4/43
22/22 - 1s - 26ms/step - accuracy: 0.6443 - loss: nan
Epoch 5/43
22/22 - 1s - 28ms/step - accuracy: 0.6443 - loss: nan
Epoch 6/43
22/22 - 1s - 25ms/step - accuracy: 0.6443 - loss: nan
Epoch 7/43
22/22 - 1s - 26ms/step - accuracy: 0.6443 - loss: nan
Epoch 8/43
22/22 - 1s - 26ms/step - accuracy: 0.6443 - loss: nan
Epoch 9/43
22/22 - 1s - 32ms/step - accuracy: 0.6443 - loss: nan
Epoch 10/43
22/22 - 1s - 24ms/step - accuracy: 0.6443 - loss: nan
Epoch 11/43
22/22 - 1s - 25ms/step - accuracy: 0.6443 - loss: nan
Epoch 12/43
22/22 - 1s - 27ms/step - accuracy: 0.6443 - loss: nan
Epoch 13/43
22/22 - 1s - 27ms/step - accuracy: 0.6443 - loss: nan
Epoch 14/43
22/22 - 1s - 27ms/step - accuracy: 0.6443 - loss: nan
Epoch 15/43
22/22 - 1s - 27ms/step - accuracy: 0.6443 - loss: nan
Epoch 16/43
22/22 

...

C:\Users\andyc\anaconda3\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
C:\Users\andyc\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


36/36 - 3s - 79ms/step - accuracy: 0.5026 - loss: 1.9769
Epoch 2/38
36/36 - 1s - 23ms/step - accuracy: 0.6442 - loss: 1.4342
Epoch 3/38
36/36 - 1s - 19ms/step - accuracy: 0.6442 - loss: 1.3037
Epoch 4/38
36/36 - 1s - 22ms/step - accuracy: 0.6442 - loss: 1.2536
Epoch 5/38
36/36 - 2s - 43ms/step - accuracy: 0.6442 - loss: 1.2282
Epoch 6/38
36/36 - 1s - 35ms/step - accuracy: 0.6442 - loss: 1.2132
Epoch 7/38
36/36 - 1s - 36ms/step - accuracy: 0.6442 - loss: 1.2034
Epoch 8/38
36/36 - 1s - 28ms/step - accuracy: 0.6442 - loss: 1.1965
Epoch 9/38
36/36 - 1s - 30ms/step - accuracy: 0.6442 - loss: 1.1914
Epoch 10/38
36/36 - 1s - 26ms/step - accuracy: 0.6442 - loss: 1.1875
Epoch 11/38
36/36 - 1s - 41ms/step - accuracy: 0.6442 - loss: 1.1845
Epoch 12/38
36/36 - 1s - 30ms/step - accuracy: 0.6442 - loss: 1.1821
Epoch 13/38
36/36 - 2s - 42ms/step - accuracy: 0.6442 - loss: 1.1801
Epoch 14/38
36/36 - 1s - 28ms/step - accuracy: 0.6442 - loss: 1.1784
Epoch 15/38
36/36 - 1s - 39ms/step - accuracy: 0.6442 

C:\Users\andyc\anaconda3\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
C:\Users\andyc\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


36/36 - 3s - 87ms/step - accuracy: 0.3012 - loss: 2.1699
Epoch 2/38
36/36 - 1s - 35ms/step - accuracy: 0.6442 - loss: 1.4903
Epoch 3/38
36/36 - 1s - 25ms/step - accuracy: 0.6442 - loss: 1.3143
Epoch 4/38
36/36 - 1s - 37ms/step - accuracy: 0.6442 - loss: 1.2538
Epoch 5/38
36/36 - 1s - 40ms/step - accuracy: 0.6442 - loss: 1.2260
Epoch 6/38
36/36 - 1s - 28ms/step - accuracy: 0.6442 - loss: 1.2104
Epoch 7/38
36/36 - 1s - 29ms/step - accuracy: 0.6442 - loss: 1.2006
Epoch 8/38
36/36 - 1s - 37ms/step - accuracy: 0.6442 - loss: 1.1939
Epoch 9/38
36/36 - 1s - 38ms/step - accuracy: 0.6442 - loss: 1.1890
Epoch 10/38
36/36 - 1s - 33ms/step - accuracy: 0.6442 - loss: 1.1853
Epoch 11/38
36/36 - 1s - 29ms/step - accuracy: 0.6442 - loss: 1.1824
Epoch 12/38
36/36 - 1s - 31ms/step - accuracy: 0.6442 - loss: 1.1801
Epoch 13/38
36/36 - 2s - 44ms/step - accuracy: 0.6442 - loss: 1.1781
Epoch 14/38
36/36 - 1s - 38ms/step - accuracy: 0.6442 - loss: 1.1765
Epoch 15/38
36/36 - 1s - 34ms/step - accuracy: 0.6442 

ValueError: Input y contains NaN.

In [62]:
optimum = nn_opt.max['params']
learning_rate = optimum['learning_rate']

activationL = ['relu', 'sigmoid', 'softplus', 'softsign', 'tanh', 'selu', 'elu', 'exponential', LeakyReLU, 'relu']
optimum['activation'] = activationL[round(optimum['activation'])]

optimum['batch_size'] = round(optimum['batch_size'])
optimum['epochs'] = round(optimum['epochs'])
optimum['layers1'] = round(optimum['layers1'])
optimum['layers2'] = round(optimum['layers2'])
optimum['neurons'] = round(optimum['neurons'])

optimizerL = ['Adam', 'SGD', 'RMSprop', 'Adadelta', 'Adagrad', 'Adamax', 'Nadam', 'Ftrl', 'Adam']
optimizerD = {
    'Adam': Adam(learning_rate=learning_rate),
    'SGD': SGD(learning_rate=learning_rate),
    'RMSprop': RMSprop(learning_rate=learning_rate),
    'Adadelta': Adadelta(learning_rate=learning_rate),
    'Adagrad': Adagrad(learning_rate=learning_rate),
    'Adamax': Adamax(learning_rate=learning_rate),
    'Nadam': Nadam(learning_rate=learning_rate),
    'Ftrl': Ftrl(learning_rate=learning_rate)
}
optimum['optimizer'] = optimizerD[optimizerL[round(optimum['optimizer'])]]
optimum

{'activation': 'selu',
 'batch_size': 929,
 'dropout': 0.5063915685439514,
 'dropout_rate': 0.131589960443724,
 'epochs': 21,
 'kernel': 1.4568184424104407,
 'layers1': 1,
 'layers2': 1,
 'learning_rate': 0.8113610465649194,
 'neurons': 57,
 'normalization': 0.36861752031667094,
 'optimizer': <keras.src.optimizers.adadelta.Adadelta at 0x18f2a2ccb30>}

# 6. Running CNN with New Parameters

In [65]:
# Setting the model with optimized hyperparameters

epochs = 47
batch_size = 460

timesteps = len(X_train[0])
input_dim = len(X_train[0][0])
n_classes = 15

layers1 = 1
layers2 = 2
activation = 'softsign'
kernel = int(round(1.9444298503238986))  # Rounded kernel size for Conv1D
neurons = 61
normalization = 0.770967179954561
dropout = 0.7296061783380641
dropout_rate = 0.19126724140656393
optimizer = Adadelta(learning_rate=0.7631771981307285)  # Instantiate RMSprop with learning rate

model = Sequential()
model.add(Conv1D(neurons, kernel_size=kernel, activation=activation, input_shape=(timesteps, input_dim)))

if normalization > 0.5:
    model.add(BatchNormalization())

for i in range(layers1):
    model.add(Dense(neurons, activation=activation))

if dropout > 0.5:
    model.add(Dropout(dropout_rate))

for i in range(layers2):
    model.add(Dense(neurons, activation=activation))

model.add(MaxPooling1D())
model.add(Flatten())
model.add(Dense(n_classes, activation='softmax')) 

model.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

C:\Users\andyc\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [67]:
model.summary()

Model: "sequential_75"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv1d_75 (Conv1D)                   │ (None, 14, 61)              │           1,159 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_35               │ (None, 14, 61)              │             244 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_410 (Dense)                    │ (None, 14, 61)              │           3,782 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_35 (Dropout)                 │ (None, 14, 61)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_411 (Dense)                    │ (None, 14, 61)              │           3,782 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_412 (Dense)                    │ (None, 14, 61)              │           3,782 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_75 (MaxPooling1D)      │ (None, 7, 61)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_75 (Flatten)                 │ (None, 427)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_413 (Dense)                    │ (None, 15)                  │           6,420 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 19,169 (74.88 KB)

 Trainable params: 19,047 (74.40 KB)

 Non-trainable params: 122 (488.00 B)

In [69]:
# Putting the y_test set back into a one-hot configuration

y_train_one_hot = to_categorical(y_train, num_classes=15)

In [71]:
# Checking both shapes

print(f'X_train shape: {X_train.shape}')
print(f'y_train_one_hot shape: {y_train_one_hot.shape}')

X_train shape: (17212, 15, 9)
y_train_one_hot shape: (17212, 15)


In [73]:
# Compiling the model with categorical_crossentropy

model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

In [75]:
# Fitting the model to the data

model.fit(X_train, y_train_one_hot, batch_size=batch_size, epochs=epochs, verbose=2)

Epoch 1/47
38/38 - 4s - 97ms/step - accuracy: 0.6014 - loss: 1.3535
Epoch 2/47
38/38 - 1s - 38ms/step - accuracy: 0.6831 - loss: 0.9219
Epoch 3/47
38/38 - 1s - 33ms/step - accuracy: 0.7143 - loss: 0.8309
Epoch 4/47
38/38 - 1s - 36ms/step - accuracy: 0.7344 - loss: 0.7754
Epoch 5/47
38/38 - 1s - 33ms/step - accuracy: 0.7511 - loss: 0.7324
Epoch 6/47
38/38 - 1s - 34ms/step - accuracy: 0.7588 - loss: 0.6976
Epoch 7/47
38/38 - 1s - 36ms/step - accuracy: 0.7703 - loss: 0.6661
Epoch 8/47
38/38 - 1s - 31ms/step - accuracy: 0.7831 - loss: 0.6397
Epoch 9/47
38/38 - 1s - 23ms/step - accuracy: 0.7914 - loss: 0.6089
Epoch 10/47
38/38 - 1s - 19ms/step - accuracy: 0.7979 - loss: 0.5843
Epoch 11/47
38/38 - 1s - 35ms/step - accuracy: 0.8059 - loss: 0.5602
Epoch 12/47
38/38 - 1s - 24ms/step - accuracy: 0.8126 - loss: 0.5428
Epoch 13/47
38/38 - 1s - 22ms/step - accuracy: 0.8193 - loss: 0.5159
Epoch 14/47
38/38 - 1s - 22ms/step - accuracy: 0.8264 - loss: 0.5011
Epoch 15/47
38/38 - 1s - 34ms/step - accura

# 7. Creating the Confusion Matrix

In [78]:
# Defining the list of stations names

stations = {
0: 'BASEL',
1: 'BELGRADE',
2: 'BUDAPEST',
3: 'DEBILT',
4: 'DUSSELDORF',
5: 'HEATHROW',
6: 'KASSEL',
7: 'LJUBLJANA',
8: 'MAASTRICHT',
9: 'MADRID',
10: 'MUNCHENB',
11: 'OSLO',
12: 'SONNBLICK',
13: 'STOCKHOLM',
14: 'VALENTIA'
}

In [80]:
def confusion_matrix(y_true, y_pred, stations):
    # Check if y_true and y_pred are one-hot encoded or already class indices
    if y_true.ndim == 1:
        y_true_labels = y_true
    else:
        y_true_labels = np.argmax(y_true, axis=1)
    
    if y_pred.ndim == 1:
        y_pred_labels = y_pred
    else:
        y_pred_labels = np.argmax(y_pred, axis=1)
        
    # Map numeric labels to activity names
    y_true_series = pd.Series([stations[y] for y in y_true_labels])
    y_pred_series = pd.Series([stations[y] for y in y_pred_labels])
    
    return pd.crosstab(y_true_series, y_pred_series, rownames=['True'], colnames=['Pred'])

In [82]:
y_pred = model.predict(X_test)

180/180 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


In [84]:
# Evaluate

print(confusion_matrix(y_test, y_pred, stations))

Pred        BASEL  BELGRADE  BUDAPEST  DEBILT  DUSSELDORF  HEATHROW  KASSEL  \
True                                                                          
BASEL        3522        67        12       5           2         2       0   
BELGRADE      116       993         1       1           0         0       0   
BUDAPEST       25        36       134       1           0         0       0   
DEBILT          7         8         8      59           1         0       0   
DUSSELDORF      5         1         3      16           8         5       0   
HEATHROW       16         5         2       3           4        61       0   
KASSEL          1         6         3       0           1         0       3   
LJUBLJANA       7         5         2       1           0         2       2   
MAASTRICHT      3         0         0       0           0         0       0   
MADRID         13        19        12       2           1        22       0   
MUNCHENB        6         1         0       0       